In [20]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("dwd")

In [21]:

import os
import re
import io
import glob
import zipfile
from urllib.parse import urljoin

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()  # loads variables from a `.env` file in the working directory
DATA_DIR = os.getenv("path_to_data")
if not DATA_DIR:
    raise RuntimeError("Missing 'path_to_data' in .env or environment.")

BASE_URL = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/"
CATEGORIES = {
    "air_temperature": "lufttemp",
    "moisture": "feuchtigkeit",
    "pressure": "druck",
    "soil_temperature": "boden",
    "solar": "strahlung",
    "sun": "sun",  # sunshine duration
}
ID_MIN = 4926
ID_MAX = 4933

# recents/ is created inside the folder provided by path_to_data
RECENT_DIR = os.path.join(DATA_DIR, "recents")
os.makedirs(RECENT_DIR, exist_ok=True)

In [22]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [str(c).strip().upper() for c in df.columns]
    return df

def pick_timestamp_column(df: pd.DataFrame) -> str | None:
    if "MESS_DATUM" in df.columns:
        return "MESS_DATUM"
    for alt in ["MESS_DATUM_BEGINN", "MESS_DATUM_BEG", "MESS_DATUM_ANFANG"]:
        if alt in df.columns:
            return alt
    return None

def fix_dates(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_columns(df.copy())
    ts_col = pick_timestamp_column(df)
    if ts_col is None:
        return df
    df["DATUM"] = df[ts_col].astype(str).str.slice(0, 10)
    df["DATUM"] = pd.to_datetime(df["DATUM"], format="%Y%m%d%H", errors="coerce")
    df.set_index("DATUM", inplace=True)
    return df

def rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    renames = {}
    return df.rename(columns=renames)

def remove_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols_to_drop = [c for c in ["EOR", "STATIONS_ID", "MESS_DATUM", "MESS_DATUM_BEGINN", "MESS_DATUM_BEG", "MESS_DATUM_ANFANG"] if c in df.columns]
    return df.drop(columns=cols_to_drop, errors="ignore")

def download_file(url: str) -> bytes:
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    return resp.content

def list_files(url: str) -> list[str]:
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    pattern = re.compile(r'href="([^"]+\.zip)"')
    return pattern.findall(resp.text)

def extract_id(name: str) -> int | None:
    m = re.search(r"_(\d{5})_", name)
    if not m:
        return None
    return int(m.group(1))

def read_zip(content: bytes) -> pd.DataFrame:
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        # pick the single data file that starts with "produkt"
        txt_files = [f for f in z.namelist() if f.lower().endswith(".txt")]
        produkt_files = [f for f in txt_files if os.path.basename(f).lower().startswith("produkt")]
        if not produkt_files:
            return pd.DataFrame()
        with z.open(sorted(produkt_files)[0]) as f:
            df = pd.read_csv(f, sep=";", encoding="latin1", dtype=str)
            return standardize_columns(df)

# -----------------
# Core update & aggregate
# -----------------
def update_category(cat: str, short: str) -> None:
    # 'solar' usually has no 'recent' dir on DWD; fall back to folder root and filter *_row.zip
    url = urljoin(BASE_URL, f"{cat}/recent/")
    use_row_only = False
    alt_url = urljoin(BASE_URL, f"{cat}/")
    try:
        files = list_files(url)
        if not files:
            files = list_files(alt_url)
            url = alt_url
            use_row_only = True
    except Exception:
        try:
            files = list_files(alt_url)
            url = alt_url
            use_row_only = True
        except Exception:
            return

    target_dir = os.path.join(RECENT_DIR, cat)
    os.makedirs(target_dir, exist_ok=True)

    all_frames = []
    for file in files:
        if use_row_only and not file.endswith("_row.zip"):
            continue
        sid = extract_id(file)
        if sid is None or sid < ID_MIN or sid > ID_MAX:
            continue
        dest_path = os.path.join(target_dir, file)
        try:
            if not os.path.exists(dest_path):
                content = download_file(urljoin(url, file))
                with open(dest_path, "wb") as fout:
                    fout.write(content)
            else:
                with open(dest_path, "rb") as f:
                    content = f.read()
        except Exception:
            continue
        df = read_zip(content)
        if not df.empty:
            all_frames.append(df)

    if not all_frames:
        return

    df_new = pd.concat(all_frames, ignore_index=True)

    # de-dup raw using available keys
    keys = [k for k in ["MESS_DATUM", "MESS_DATUM_BEGINN", "MESS_DATUM_BEG", "STATIONS_ID"] if k in df_new.columns]
    if keys:
        df_new = df_new.drop_duplicates(subset=keys)

    # RAW file written into DATA_DIR (parent folder from .env)
    raw_path = os.path.join(DATA_DIR, f"schnarrenberg_dwd_{short}.csv")
    if os.path.exists(raw_path):
        df_exist = pd.read_csv(raw_path, sep=";", dtype=str)
        df_exist = standardize_columns(df_exist)
        df_new = pd.concat([df_exist, df_new], ignore_index=True)
        keys = [k for k in ["MESS_DATUM", "MESS_DATUM_BEGINN", "MESS_DATUM_BEG", "STATIONS_ID"] if k in df_new.columns]
        if keys:
            df_new.drop_duplicates(subset=keys, inplace=True)
    df_new.to_csv(raw_path, sep=";", index=False)

    # CLEAN file also in DATA_DIR (NOT in recents/)
    df_clean = fix_dates(df_new.copy())
    df_clean = rename_columns(df_clean)
    df_clean = remove_duplicate_columns(df_clean)
    clean_path = os.path.join(DATA_DIR, f"clean_schnarrenberg_dwd_{short}.csv")
    if os.path.exists(clean_path):
        df_exist = pd.read_csv(clean_path)
        df_clean = pd.concat([df_exist, df_clean.reset_index()], ignore_index=True)
    if "DATUM" in df_clean.columns:
        df_clean.drop_duplicates(subset="DATUM", inplace=True)
    df_clean.to_csv(clean_path, index=False)

def aggregate_all() -> None:
    files = glob.glob(os.path.join(DATA_DIR, "clean_schnarrenberg_dwd_*.csv"))
    date_range = pd.date_range("2023-01-01", "2025-12-31 23:00:00", freq="h")
    df_agg = pd.DataFrame(index=date_range)
    for file in files:
        df_part = pd.read_csv(file)
        if "DATUM" not in df_part.columns:
            continue
        df_part["DATUM"] = pd.to_datetime(df_part["DATUM"], errors="coerce")
        df_part.dropna(subset=["DATUM"], inplace=True)
        df_part.set_index("DATUM", inplace=True)
        df_part = remove_duplicate_columns(df_part)
        df_agg = df_agg.merge(df_part, how="left", left_index=True, right_index=True, suffixes=("", "_y"))
    df_agg.dropna(how="all", inplace=True)
    # OVERWRITE the aggregated file in the parent folder from .env (not in recents/)
    df_agg.to_csv(os.path.join(DATA_DIR, "clean_wetter_komplett.csv"))

In [23]:
# -----------------
if __name__ == "__main__":
    for cat, short in CATEGORIES.items():
        update_category(cat, short)
    aggregate_all()

C:\Users\Dome Arbeit\AppData\Local\Temp\ipykernel_1888\1355514866.py:133: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_exist = pd.read_csv(clean_path)
C:\Users\Dome Arbeit\AppData\Local\Temp\ipykernel_1888\1355514866.py:144: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_part = pd.read_csv(file)
